In [2]:
import os
from pathlib import Path
import shutil

from ultralytics.utils.downloads import download
from ultralytics.utils import ASSETS_URL, TQDM


def visdrone2yolo(dir, split, source_name=None):
    """Convert VisDrone annotations to YOLO format with images/{split} and labels/{split} structure."""
    from PIL import Image

    source_dir = dir / (source_name or f"VisDrone2019-DET-{split}")
    images_dir = dir / "images" / split
    labels_dir = dir / "labels" / split
    labels_dir.mkdir(parents=True, exist_ok=True)

    # Move images to new structure
    if (source_images_dir := source_dir / "images").exists():
        images_dir.mkdir(parents=True, exist_ok=True)
        for img in source_images_dir.glob("*.jpg"):
            img.rename(images_dir / img.name)

    for f in TQDM((source_dir / "annotations").glob("*.txt"), desc=f"Converting {split}"):
        img_size = Image.open(images_dir / f.with_suffix(".jpg").name).size
        dw, dh = 1.0 / img_size[0], 1.0 / img_size[1]
        lines = []

        with open(f, encoding="utf-8") as file:
            for row in [x.split(",") for x in file.read().strip().splitlines()]:
                if row[4] != "0":  # Skip ignored regions
                    x, y, w, h = map(int, row[:4])
                    cls = int(row[5]) - 1
                    # Convert to YOLO format
                    x_center, y_center = (x + w / 2) * dw, (y + h / 2) * dh
                    w_norm, h_norm = w * dw, h * dh
                    lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

        (labels_dir / f.name).write_text("".join(lines), encoding="utf-8")


# Download (ignores test-challenge split)
dir = Path('VisDrone/')  # dataset root dir
urls = [
    f"{ASSETS_URL}/VisDrone2019-DET-train.zip",
    f"{ASSETS_URL}/VisDrone2019-DET-val.zip",
    f"{ASSETS_URL}/VisDrone2019-DET-test-dev.zip",
    # f"{ASSETS_URL}/VisDrone2019-DET-test-challenge.zip",
]
download(urls, dir=dir, threads=4)

Unzipping VisDrone/VisDrone2019-DET-val.zip to /home/gpu_03/class-agnostic/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 4.7Kfiles/s 0.2s0.0s.3ss<7.0s
Unzipping VisDrone/VisDrone2019-DET-test-dev.zip to /home/gpu_03/class-agnostic/VisDrone/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 2.9Kfiles/s 1.1s0.9s
Unzipping VisDrone/VisDrone2019-DET-train.zip to /home/gpu_03/class-agnostic/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 3.8Kfiles/s 3.4s<0.0s


In [3]:
# Convert
splits = {"VisDrone2019-DET-train": "train", "VisDrone2019-DET-val": "val", "VisDrone2019-DET-test-dev": "test"}
for folder, split in splits.items():
    visdrone2yolo(dir, split, folder)  # convert VisDrone annotations to YOLO labels
    shutil.rmtree(dir / folder)  # cleanup original directory


Converting train: ━━━━━━━━━━━━ 6471 5.8Kit/s 1.2s
Converting val: ━━━━━━━━━━━━ 548 1.8Kit/s 0.2s
Converting test: ━━━━━━━━━━━━ 1610 4.1Kit/s 0.3s
